<p>
    <b>ADA Facility report</b><br>
    <b>Gulbenkian Institute for Molecular Medicine</b><br>
    <b>Oeiras, PT</b>
</p>
<br>

# 1. Database Import

To make these sequences more "stable", we are working locally with reference files from the latest version of reference genome from *Mus musculus* provided by ENSEMBL (version 116).

* `.gtf` file - https://ftp.ensembl.org/pub/release-116/gtf/mus_musculus/Mus_musculus.GRCm39.116.gtf.gz
* `.fasta` file - https://ftp.ensembl.org/pub/release-116/fasta/mus_musculus/dna/Mus_musculus.GRCm39.dna.primary_assembly.fa.gz

To work with the reference files from ENSEMBL, we are using the `pyranges` package (https://pyranges.readthedocs.io/en/latest/index.html)

In [1]:
#| label: setup
#| code-fold: true
#| code-summary: "Setup: Imports and Reference File Paths"
#| output: false
import pyranges as pr
import pyfaidx
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
import os
import session_info

# -------- Setting the Path for the necessary files -------- #

GTF_v116 = "/srv/ext_home/asbarros/genomes/mmusculus/Ensembl_v116/Mus_musculus.GRCm39.116.gtf.gz"
FA_v116  = "/srv/ext_home/asbarros/genomes/mmusculus/Ensembl_v116/Mus_musculus.GRCm39.dna.primary_assembly.fa"

# -------- Reference Genome (.fasta) -------- #
#fa  = pyfaidx.Fasta(FA_v116)                                                 # Index the fasta file (creating the .fai file) - Only requires to be done one time

In [2]:
#| label: gtf_import
#| code-fold: true
#| code-summary: "Import and Filter Reference GTF"
#| output: false
# -------- Reference GTF -------- #
gtf = pr.read_gtf(GTF_v116, duplicate_attr=True)                              # Import the GTF file, which has all the information necessary to extract the coordinates. Keep all attributes in single string (e.g. keep tag 'Ensembl_canonical')
gtf_canonical = gtf[(gtf.Feature == "transcript") & 
                     gtf.tag.str.contains("Ensembl_canonical", na=False)]     # Filter to include only transcripts that contain the tag "Ensembl_canonical" (which corresponds to canonical transcripts)

# 2. Analysis of SpiC-Regulated Genes

To determine whether SpiC regulates its target genes directly or indirectly, promoter sequences were retrieved for the following genes:

*Nfe2l2, Blvra, Hmox1, Vcam1, Slc40a1, Nqo1, Gsr, Slc7a11, Gclc, Cat, Gpx4, Gclm, Gss, Txnrd1, Fth1, Nfs1, Cisd1, Aifm2, Acsl4, Acsl2, LPCAT3, G6PDX, PGD, ME1, IDH1, Nox1, Elovl7, Ido1, ALAS1, DHODH, ACSL3, SCD1,* and *SCD5*.

The promoter region for each gene was defined as **−2,000 bp to +200 bp relative to the transcription start site (TSS)** of the canonical transcript. This interval was selected because it encompasses the proximal regulatory regions where transcription factor binding is most frequently observed.

More specifically:

- The **−2,000 bp upstream region** captures the proximal promoter and many of the cis-regulatory transcription factor binding sites.
- The **+200 bp downstream region** includes the TSS-proximal sequence, which often contains core promoter elements and regulatory motifs that overlap the transcription start site.
- The **TSS of the canonical transcript** was used as the reference coordinate to ensure a consistent and standardized definition of promoter regions across all analyzed genes.

These promoter sequences were subsequently used for transcription factor binding site analysis to evaluate the potential for direct SpiC-mediated transcriptional regulation, or indirect regulation through Nfe2l2-mediated transcriptional regulation.

## 2.1 Extract transcript coordinates

In [3]:
#| label: gene_subset
#| code-fold: true
#| code-summary: "Select Canonical Transcripts for Genes of Interest"
#| output: false
                                        # List of genes of interest
genes = ["Nfe2l2", "Spic", "Blvra", "Hmox1",  "Vcam1", "Slc40a1", "Nqo1", "Gsr", "Slc7a11", "Gclc", "Cat", "Gpx4", "Gclm", "Gss", "Txnrd1",  "Fth1", "Nfs1", "Cisd1", 
         "Aifm2", "Acsl4", "Acsl2", "LPCAT3", "G6PDX", "PGD", "ME1", "IDH1", "Nox1", "Elovl7", "Ido1", "ALAS1", "FSP1 (AIFM2)", "DHODH",
        "ACSL3", "SCD1", "SCD5"]             

gtf_sub = gtf_canonical[gtf_canonical.gene_name.isin(genes)]      # Using the subset for canonical transcripts, select the ones for the genes of interest.

print(gtf_sub.df['gene_name'].duplicated().any())                 # Check if more than one transcript was given per gene_name; if this is true - check the object

False


Register the Start & End coordinates

In [4]:
#| label: gtf_sub_preview
#| code-fold: true
#| code-summary: "Selected Transcript Coordinates"
#| output: asis
gtf_sub    # Inspect the selected canonical transcripts (one per gene of interest)


/srv/ext_home/ssalazar/.local/lib/python3.12/site-packages/pyranges/tostring2.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([plus, minus])
/srv/ext_home/ssalazar/.local/lib/python3.12/site-packages/pyranges/tostring2.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([plus, minus])
/srv/ext_home/ssalazar/.local/lib/python3.12/site-packages/pyranges/tostring2.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprec

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_version,...,transcript_source,transcript_biotype,tag,transcript_support_level,exon_number,exon_id,exon_version,ccds_id,protein_id,protein_version
0,1,ensembl_havana,transcript,45947210,45964818,.,-,.,ENSMUSG00000025993,12,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 10),NaN,NaN,NaN,CCDS14933,NaN,NaN
1,2,ensembl_havana,transcript,126912566,126939017,.,+,.,ENSMUSG00000001999,17,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 14),NaN,NaN,NaN,CCDS16692,NaN,NaN
2,2,ensembl_havana,transcript,103284193,103315505,.,-,.,ENSMUSG00000027187,12,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 9),NaN,NaN,NaN,CCDS16478,NaN,NaN
3,2,ensembl_havana,transcript,155405095,155434730,.,-,.,ENSMUSG00000027610,19,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 12),NaN,NaN,NaN,CCDS16951,NaN,NaN
4,2,ensembl_havana,transcript,155965555,155986374,.,-,.,ENSMUSG00000027618,19,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 15),NaN,NaN,NaN,CCDS16962,NaN,NaN
5,2,ensembl_havana,transcript,75504738,75534985,.,-,.,ENSMUSG00000015839,8,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 4),NaN,NaN,NaN,CCDS16150,NaN,NaN
6,3,ensembl_havana,transcript,122039125,122065061,.,+,.,ENSMUSG00000028124,18,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 13),NaN,NaN,NaN,CCDS17808,NaN,NaN
7,3,ensembl_havana,transcript,50319386,50398063,.,-,.,ENSMUSG00000027737,12,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 5),NaN,NaN,NaN,CCDS17335,NaN,NaN
8,3,ensembl_havana,transcript,115903597,115923347,.,-,.,ENSMUSG00000027962,16,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 12),NaN,NaN,NaN,CCDS17785,NaN,NaN
9,8,ensembl_havana,transcript,34143265,34188191,.,+,.,ENSMUSG00000031584,18,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 8),NaN,NaN,NaN,CCDS22235,NaN,NaN


Plus strand (+): transcription goes left→right along the reference. The biological start (TSS, 5′ end) is at Start, and the 3′ end is at End.
Minus strand (-): transcription goes right→left along the reference. The biological start (TSS, 5′ end) is at End, and the 3′ end is at Start.

## 2.2 Extract promoter coordinates

In [5]:
#| label: promoter_coords
#| code-fold: true
#| code-summary: "Compute 2 kb Promoter Coordinates"
#| output: false
prom_2kb = gtf_sub.five_end().slack({"5": 2000, "3": 200})                             # 2kb promoters. Strand-Aware function, that will change the columns 'Start' & 'End' in the PyRanges object.

display(Markdown(f'**prom_2kb is a PyRanges?** {isinstance(prom_2kb, pr.PyRanges)}'))  # Confirm that the resulting object is indeed a PyRanges object

**prom_2kb is a PyRanges?** True

In [6]:
#| label: promoter_below_zero_check
#| code-fold: true
#| code-summary: "Sanity Check: Promoter Start Not Below Zero"
#| output: false

#**Sanity Check:** Is the starting position of the promoters below zero? (gene may start at the beginning of the chromosome)
display(Markdown(f'**Any promoter starts below zero?** {(prom_2kb.df['Start']<0).any()}'))  # Confirm that the resulting object is indeed a PyRanges object

**Any promoter starts below zero?** False

Register the Start & End coordinates. 

* In the *strand -* the TSS is at the `End` coordinate of the transcript. Therefore the start of the promoter should be +2000bp from `End` in `gtf_sub`/transcript (Start of the promoter is in the `End` coordinate of the promoter),  and -200bp (End of the promoter is in the `Start` coordinate of the promoter). 

* In the *strand +* the TSS is at the `Start` coordinate of the transcript. Therefore the promoter should `Start` -2000bp from `Start` in `gtf_sub`/transcript (Start of the promoter is in the `Start` coordinate of the promoter),  and  -200bp (End of the promoter is in the `End` coordinate of the promoter). 



In [7]:
#| label: promoter_preview
#| code-fold: true
#| code-summary: "Promoter Coordinates"
#| output: asis
prom_2kb    # Inspect the strand-aware promoter coordinates (Start/End updated by slack)


/srv/ext_home/ssalazar/.local/lib/python3.12/site-packages/pyranges/tostring2.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([plus, minus])
/srv/ext_home/ssalazar/.local/lib/python3.12/site-packages/pyranges/tostring2.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([plus, minus])
/srv/ext_home/ssalazar/.local/lib/python3.12/site-packages/pyranges/tostring2.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprec

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_version,...,transcript_source,transcript_biotype,tag,transcript_support_level,exon_number,exon_id,exon_version,ccds_id,protein_id,protein_version
0,1,ensembl_havana,transcript,45964617,45966818,.,-,.,ENSMUSG00000025993,12,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 10),NaN,NaN,NaN,CCDS14933,NaN,NaN
1,2,ensembl_havana,transcript,126910566,126912767,.,+,.,ENSMUSG00000001999,17,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 14),NaN,NaN,NaN,CCDS16692,NaN,NaN
2,2,ensembl_havana,transcript,103315304,103317505,.,-,.,ENSMUSG00000027187,12,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 9),NaN,NaN,NaN,CCDS16478,NaN,NaN
3,2,ensembl_havana,transcript,155434529,155436730,.,-,.,ENSMUSG00000027610,19,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 12),NaN,NaN,NaN,CCDS16951,NaN,NaN
4,2,ensembl_havana,transcript,155986173,155988374,.,-,.,ENSMUSG00000027618,19,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 15),NaN,NaN,NaN,CCDS16962,NaN,NaN
5,2,ensembl_havana,transcript,75534784,75536985,.,-,.,ENSMUSG00000015839,8,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 4),NaN,NaN,NaN,CCDS16150,NaN,NaN
6,3,ensembl_havana,transcript,122037125,122039326,.,+,.,ENSMUSG00000028124,18,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 13),NaN,NaN,NaN,CCDS17808,NaN,NaN
7,3,ensembl_havana,transcript,50397862,50400063,.,-,.,ENSMUSG00000027737,12,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 5),NaN,NaN,NaN,CCDS17335,NaN,NaN
8,3,ensembl_havana,transcript,115923146,115925347,.,-,.,ENSMUSG00000027962,16,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 12),NaN,NaN,NaN,CCDS17785,NaN,NaN
9,8,ensembl_havana,transcript,34141265,34143466,.,+,.,ENSMUSG00000031584,18,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 8),NaN,NaN,NaN,CCDS22235,NaN,NaN


## 2.3 Extract sequences based on the coordinates

In [8]:
#| label: extract_sequences
#| code-fold: true
#| code-summary: "Extract Promoter Sequences from Reference"
#| output: false
seqs_2kb = pr.get_sequence(prom_2kb, path=FA_v116)                        # Based on the coordinates identified before, we get the sequences from the reference genome
#get_sequence is also strand-aware and reverse-complements the minus strand

## 2.4. Saving the sequences into a `fasta` file

In [9]:
#| label: build_sequence_frame
#| code-fold: true
#| code-summary: "Assemble Sequence Dataframe"
#| output: false
df_2kb                 = seqs_2kb.to_frame(name="DNA_Sequence")   # Convert the extracted sequences into a dataframe
df_2kb["Gene_Name"]    = prom_2kb.df['gene_name']                 # Attach gene names for use in the FASTA headers
df_2kb["TranscriptID"] = prom_2kb.df["transcript_id"]             # Attach transcript IDs for use in the FASTA headers


In [10]:
#| label: write_fasta
#| code-fold: true
#| code-summary: "Write Promoter Sequences to FASTA"
#| output: false
with open("/srv/ext_home/ssalazar/projects/GIMM_projects/DNA_018_PromotorSeq_MotifFinder/Results/promoters_SpiC_assumed_network.fasta", "w") as file:
    for index, row in df_2kb.iterrows():
        # Combine your metadata into a single header line
        header = f">{row['Gene_Name']}|{row['TranscriptID']}"
        
        # Write the header and the sequence
        file.write(f"{header}\n")
        file.write(f"{row['DNA_Sequence']}\n")

**`Fasta` files produced!**

---

# 3. Analysis of Transcription Factors Cross regulation

To investigate whether **SpiC** and **Nfe2l2** may directly regulate each other's transcription, the promoter regions of both genes were retrieved and analyzed. Promoter regions were defined as described above, comprising **−2,000 bp to +200 bp relative to the transcription start site (TSS)** of the canonical transcript. 


## 3.1 Extract transcript coordinates

In [29]:
#| label: gene_subset_ft
#| code-fold: true
#| code-summary: "Select SpiC / Nfe2l2 Canonical Transcripts"
#| output: false
                                        # List of genes of interest
genes_ft = ["Nfe2l2", "Spic", "Bach1"]  
gene_regulated_bach = ["Nqo1"]           
gtf_sub_ft = gtf_canonical[gtf_canonical.gene_name.isin(genes_ft)]      # Using the subset for canonical transcripts, select the ones for the genes of interest.
gtf_sub_bach = gtf_canonical[gtf_canonical.gene_name.isin(gene_regulated_bach)]  
print(gtf_sub_ft.df['gene_name'].duplicated().any())                 # Check if more than one transcript was given per gene_name; if this is true - check the object
print(gtf_sub_bach.df['gene_name'].duplicated().any())       

False
False


Register the Start & End coordinates

In [30]:
#| label: gtf_sub_ft_preview
#| code-fold: true
#| code-summary: "Selected Transcript Coordinates (TF Cross-Regulation)"
#| output: asis
gtf_sub_ft    # Inspect the selected SpiC / Nfe2l2 canonical transcripts


,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_version,...,transcript_source,transcript_biotype,tag,transcript_support_level,exon_number,exon_id,exon_version,ccds_id,protein_id,protein_version
0,2,ensembl_havana,transcript,75504738,75534985,.,-,.,ENSMUSG00000015839,8,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 4),NaN,NaN,NaN,CCDS16150,NaN,NaN
1,10,ensembl_havana,transcript,88510633,88518885,.,-,.,ENSMUSG00000004359,18,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 14),NaN,NaN,NaN,CCDS36023,NaN,NaN
2,16,ensembl_havana,transcript,87495781,87530234,.,+,.,ENSMUSG00000025612,7,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 5),NaN,NaN,NaN,CCDS28293,NaN,NaN


In [31]:
#| label: gtf_sub_bachNqo1_preview
#| code-fold: true
#| code-summary: "Selected Transcript Coordinates of Nqo1"
#| output: asis

gtf_sub_bach

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_version,...,transcript_source,transcript_biotype,tag,transcript_support_level,exon_number,exon_id,exon_version,ccds_id,protein_id,protein_version
4497476,8,ensembl_havana,transcript,108113834,108129934,.,-,.,ENSMUSG00000003849,10,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 8),NaN,NaN,NaN,CCDS40465,NaN,NaN


## 3.2 Extract promoter coordinates

In [32]:
#| label: promoter_coords_ft
#| code-fold: true
#| code-summary: "Compute Promoter Coordinates (TFs)"
#| output: false
prom_2kb_ft = gtf_sub_ft.five_end().slack({"5": 2000, "3": 200})                             # 2kb promoters. Strand-Aware function, that will change the columns 'Start' & 'End' in the PyRanges object.
prom_2kb_nqo1 = gtf_sub_bach.five_end().slack({"5": 2000, "3": 200}) 
display(Markdown(f'**prom_2kb_ft is a PyRanges?** {isinstance(prom_2kb_ft, pr.PyRanges)}'))  # Confirm that the resulting object is indeed a PyRanges object
display(Markdown(f'**prom_2kb_ft is a PyRanges?** {isinstance(prom_2kb_nqo1, pr.PyRanges)}')) 

**prom_2kb_ft is a PyRanges?** True

**prom_2kb_ft is a PyRanges?** True

In [33]:
#| label: promoter_below_zero_check_ft
#| code-fold: true
#| code-summary: "Sanity Check: Promoter Start Not Below Zero (TFs)"
#| output: false

#**Sanity Check:** Is the starting position of the promoters below zero? (gene may start at the beginning of the chromosome)
display(Markdown(f'**Any promoter starts below zero?** {(prom_2kb_ft.df['Start']<0).any()}'))  # Confirm that the resulting object is indeed a PyRanges object
display(Markdown(f'**Any promoter starts below zero?** {(prom_2kb_nqo1.df['Start']<0).any()}')) 

**Any promoter starts below zero?** False

**Any promoter starts below zero?** False

Register the Start & End coordinates. 

* The `Start` coordinates should be -2000bp from `Start` in `gtf_sub` when gene is *strand +* or +200bp when *strand is -*.

* The `End` coordinates should be +200bp from `Start` in `gtf_sub` when gene is *strand +* or -2000bp when *strand is -*.

In [34]:
#| label: promoter_preview_ft
#| code-fold: true
#| code-summary: "Promoter Coordinates (TFs)"
#| output: asis
prom_2kb_ft    # Inspect the strand-aware promoter coordinates (TF cross-regulation)


,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_version,...,transcript_source,transcript_biotype,tag,transcript_support_level,exon_number,exon_id,exon_version,ccds_id,protein_id,protein_version
0,2,ensembl_havana,transcript,75534784,75536985,.,-,.,ENSMUSG00000015839,8,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 4),NaN,NaN,NaN,CCDS16150,NaN,NaN
1,10,ensembl_havana,transcript,88518684,88520885,.,-,.,ENSMUSG00000004359,18,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 14),NaN,NaN,NaN,CCDS36023,NaN,NaN
2,16,ensembl_havana,transcript,87493781,87495982,.,+,.,ENSMUSG00000025612,7,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 5),NaN,NaN,NaN,CCDS28293,NaN,NaN


In [35]:
#| label: promoter_preview_nqo1
#| code-fold: true
#| code-summary: "Promoter Coordinates Nqo1"
#| output: asis
prom_2kb_nqo1    # Inspect the strand-aware promoter coordinates Nqo1


,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_version,...,transcript_source,transcript_biotype,tag,transcript_support_level,exon_number,exon_id,exon_version,ccds_id,protein_id,protein_version
0,8,ensembl_havana,transcript,108129733,108131934,.,-,.,ENSMUSG00000003849,10,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 8),NaN,NaN,NaN,CCDS40465,NaN,NaN


## 3.3 Extract sequences based on the coordinates

In [36]:
#| label: extract_sequences_ft
#| code-fold: true
#| code-summary: "Extract Promoter Sequences (TFs)"
#| output: false
seqs_2kb_ft = pr.get_sequence(prom_2kb_ft, path=FA_v116)                        # Based on the coordinates identified before, we get the sequences from the reference genome
seqs_2kb_nqo1 = pr.get_sequence(prom_2kb_nqo1, path=FA_v116)  

## 3.4. Saving the sequences into a `fasta` file

In [37]:
#| label: build_sequence_frame_ft
#| code-fold: true
#| code-summary: "Assemble Sequence Dataframe (TFs)"
#| output: false
df_2kb_ft                 = seqs_2kb_ft.to_frame(name="DNA_Sequence")   # Convert the extracted sequences into a dataframe
df_2kb_ft["Gene_Name"]    = prom_2kb_ft.df['gene_name']                 # Attach gene names for use in the FASTA headers
df_2kb_ft["TranscriptID"] = prom_2kb_ft.df["transcript_id"]             # Attach transcript IDs for use in the FASTA headers

df_2kb_nqo1                 = seqs_2kb_nqo1.to_frame(name="DNA_Sequence")   # Convert the extracted sequences into a dataframe
df_2kb_nqo1["Gene_Name"]    = prom_2kb_nqo1.df['gene_name']                 # Attach gene names for use in the FASTA headers
df_2kb_nqo1["TranscriptID"] = prom_2kb_nqo1.df["transcript_id"]             # Attach transcript IDs for use in the FASTA headers

In [47]:
#| label: write_fasta_ft
#| code-fold: true
#| code-summary: "Write Promoter Sequences to FASTA (TFs)"
#| output: false
with open("/srv/ext_home/ssalazar/projects/GIMM_projects/DNA_018_PromotorSeq_MotifFinder/Results/promoters_SpiC_Nfe2l2_Bach_Fts.fasta", "w") as file:
    for index, row in df_2kb_ft.iterrows():
        # Combine your metadata into a single header line
        header = f">{row['Gene_Name']}|{row['TranscriptID']}"
        
        # Write the header and the sequence
        file.write(f"{header}\n")
        file.write(f"{row['DNA_Sequence']}\n")

In [38]:
#| label: write_fasta_ft
#| code-fold: true
#| code-summary: "Write Promoter Sequences to FASTA (TFs)"
#| output: false
with open("/srv/ext_home/ssalazar/projects/GIMM_projects/DNA_018_PromotorSeq_MotifFinder/Results/promoters_Nqo1_BACHregulated_genes.fasta", "w") as file:
    for index, row in df_2kb_nqo1.iterrows():
        # Combine your metadata into a single header line
        header = f">{row['Gene_Name']}|{row['TranscriptID']}"
        
        # Write the header and the sequence
        file.write(f"{header}\n")
        file.write(f"{row['DNA_Sequence']}\n")

# 4. Search motifs in the promoter

## 4.1. Motif scanning with local RSAT

Motif scanning was performed with the **Regulatory Sequence Analysis Tools (RSAT)** suite to assess the presence of **SpiC** and **Nfe2l2** transcription factor binding motifs within the retrieved promoter sequences. Detection of these motifs provides preliminary evidence for potential direct regulatory interactions between the transcription factors and their target genes.

The scan is executed from this notebook against a local RSAT installation (`rsat-core`, bioconda), replacing the earlier manual Web-form procedure described in **SOP RSAT Tools - Part 2**. Running it here means the matrices, background model and thresholds are recorded in the report itself rather than in screenshots.

Position-specific scoring matrices were taken from the **CIS-BP** *Mus musculus* collection distributed with RSAT (`motif_databases`, CIS-BP 2.00). The background model is a **first-order Markov chain estimated from the input promoter set**: because a PSSM treats positions independently, dinucleotide composition is accounted for solely in the background term, and the SpiC motif consists largely of purine-purine steps that are over-represented in mouse genomic sequence. An order-0 model would not represent this and would assign such matches anticonservative *P*-values.

In [ ]:
#| label: rsat_setup
#| code-fold: true
#| code-summary: "Setup: RSAT paths, motif accessions and scan parameters"
#| output: false
import re
import shutil
import subprocess
from pathlib import Path

# -------- Paths -------- #
PROJECT   = Path("/srv/ext_home/ssalazar/projects/GIMM_projects/DNA_018_PromotorSeq_MotifFinder")
RESULTS   = PROJECT / "Results"
SCAN_DIR  = RESULTS / "matrix_scan"                    # matrices, background models, raw scans
SCAN_DIR.mkdir(parents=True, exist_ok=True)

# CIS-BP Mus musculus collection (CIS-BP 2.00), from rsa-tools/motif_databases
CISBP_MOUSE = PROJECT / "Data" / "cisBP_Mus_musculus_2019.tf"

# -------- Motifs -------- #
# Both Nfe2l2 matrices carry ID 'Nfe2l2' in the collection, so the extraction step
# below rewrites each ID to <name>_<accession>. Without that, matrix-scan reports
# ft_name='Nfe2l2' for both and the two motifs cannot be told apart in the output.
MOTIFS = ["M03009_2.00",    # Spic   -- 14 col; the informative one (M01480_2.00 is 9 col)
          "M08077_2.00",    # Nfe2l2 -- 15 col
          "M08816_2.00"]    # Nfe2l2 -- 14 col

# -------- Scan parameters -------- #
PVAL_THRESHOLD = 1e-4      # recommended upper threshold on site P-value
MARKOV_ORDER   = 1         # first-order background, estimated from the input set
BG_OLIGO_LEN   = MARKOV_ORDER + 1   # RSAT derives order m from oligos of length k = m+1
BG_STRAND      = "-1str"   # strand handling for background estimation
MATRIX_PSEUDO  = 1         # matrix pseudo-count (RSAT default)
BG_PSEUDO      = 0.01      # background pseudo-frequency; see note below
SCAN_STRANDS   = "-2str"   # both strands: section 4.2 derives genomic strand from D/R

# NOTE on -bg_pseudo: under -bginput RSAT computes this automatically from sequence
# length, but under -bgfile it defaults to 0.01. Since create-background-model is
# broken in this rsat-core build (see below) we are on the -bgfile path, so the value
# is set explicitly here to keep it on the record.

RSAT = shutil.which("rsat")
if RSAT is None:
    raise RuntimeError("`rsat` not on PATH -- activate the conda env: conda activate rsat")
print("rsat:", RSAT)


def run_rsat(subcommand, args, capture=True):
    """Call an RSAT subcommand and fail loudly.

    rsat-core exposes tools as subcommands of a single dispatcher (`rsat matrix-scan`),
    not as standalone binaries. Some tools also report errors on stdout while still
    exiting 0, so stderr text is surfaced rather than trusted to the return code.
    """
    cmd = [RSAT, subcommand] + [str(a) for a in args]
    proc = subprocess.run(cmd, capture_output=capture, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)}\n{proc.stderr or proc.stdout}")
    if capture and proc.stderr and "Error" in proc.stderr:
        raise RuntimeError(f"{' '.join(cmd)}\n{proc.stderr}")
    return proc.stdout

In [ ]:
#| label: rsat_extract_matrices
#| code-fold: true
#| code-summary: "Extract the selected CIS-BP matrices"
#| output: false

def extract_matrices(collection_path, accessions, out_path, unique_ids=True):
    """Pull selected matrices out of a multi-matrix TRANSFAC file.

    Splits on the '//' record separator and keeps blocks whose AC line matches one
    of `accessions`. With unique_ids, the ID/DE lines are rewritten as
    <name>_<accession> so every matrix yields a distinct ft_name in the scan output.

    Raises if any requested accession is absent: a silently missing matrix would
    surface much later as "no hits" and read as a biological result.
    """
    text = Path(collection_path).read_text()
    wanted, found, blocks = set(accessions), {}, []

    for block in text.split("//"):
        m = re.search(r"^AC\s+(\S+)\s*$", block, re.M)
        if not m or m.group(1) not in wanted:
            continue
        acc = m.group(1)
        nm = re.search(r"^ID\s+(.+)$", block, re.M)
        name = nm.group(1).strip() if nm else acc
        label = f"{name}_{acc}" if unique_ids else name
        if unique_ids:
            block = re.sub(r"^ID\s+.+$", f"ID  {label}", block, count=1, flags=re.M)
            block = re.sub(r"^DE\s+.+$", f"DE  {label}", block, count=1, flags=re.M)
        found[acc] = label
        blocks.append(block.strip() + "\n//\n")

    missing = wanted - set(found)
    if missing:
        raise ValueError(f"accession(s) not found in {collection_path.name}: {sorted(missing)}")

    Path(out_path).write_text("".join(blocks))
    return found


MATRIX_FILE = SCAN_DIR / "motifs_SpiC_Nfe2l2.tf"
motif_labels = extract_matrices(CISBP_MOUSE, MOTIFS, MATRIX_FILE)

# Confirm RSAT parses what we wrote, and record the consensus of each motif
consensus = run_rsat("convert-matrix", ["-i", MATRIX_FILE, "-from", "transfac",
                                       "-to", "tab", "-return", "consensus"])
print(f"{len(motif_labels)} matrices -> {MATRIX_FILE.name}")
for acc, label in motif_labels.items():
    print(f"  {acc:<14} {label}")
print()
print("\n".join(l for l in consensus.splitlines() if "consensus" in l))

In [ ]:
#| label: rsat_background
#| code-fold: true
#| code-summary: "Build the first-order background model from the promoter set"
#| output: false

def build_background(fasta_path, out_path, oligo_len=BG_OLIGO_LEN, strand=BG_STRAND):
    """Estimate a Markov background of order oligo_len-1 from a FASTA set.

    Uses oligo-analysis rather than create-background-model, which in this
    rsat-core build (2025.04.04, bioconda) writes a header-only file: the residue
    frequencies come back empty, so matrix-scan divides by a zero-sum prior and
    dies with "the sum must be strictly positive". oligo-analysis emits the same
    four-column format that -bgfile consumes, and counts correctly.

    The model is estimated from the pooled promoter set, so each sequence is scored
    against the composition of the whole submitted set rather than its own -- which
    is what the Web-form -bginput option did, and is worth keeping in mind given
    that mouse promoters are GC-rich and vary between loci.
    """
    run_rsat("oligo-analysis",
             ["-i", fasta_path, "-l", oligo_len, strand,
              "-return", "occ,freq", "-o", out_path])
    rows = [l for l in Path(out_path).read_text().splitlines()
            if l.strip() and not l.startswith((";", "#"))]
    if not rows:
        raise RuntimeError(f"background model is empty: {out_path}")
    return len(rows)


FASTA_SPIC = RESULTS / "promoters_SpiC_assumed_network.fasta"
BG_SPIC    = SCAN_DIR / f"bg_markov{MARKOV_ORDER}_SpiC_assumed_network.tab"

n = build_background(FASTA_SPIC, BG_SPIC)
print(f"background: {n} rows (expect {4 ** BG_OLIGO_LEN} for k={BG_OLIGO_LEN}) -> {BG_SPIC.name}")
print(Path(BG_SPIC).read_text().split("#seq")[-1][:400])

In [ ]:
#| label: rsat_matrix_scan
#| code-fold: true
#| code-summary: "Run matrix-scan"
#| output: false

def matrix_scan(fasta_path, matrix_path, bg_path, out_path):
    """Scan sequences with the PSSMs and write RSAT .ft output.

    Parameters chosen to match the original Web-form run, because section 4.2
    depends on them:

    -origin -0   positions are negative, measured from the END of each sequence.
                 The promoter is -2000..+200 around the TSS, so the TSS sits at
                 -201; the coordinate cells downstream add +201 to recover
                 TSS-relative positions. Using -origin start would silently shift
                 every genomic coordinate in the published table.
    -return      sites (the hits), pval (Pval/ln_Pval/sig columns) and limits
                 (the START_END rows that 4.2 filters out via ft_type == 'site').
    -2str        both strands, reported as D/R in the strand column, which 4.2
                 combines with the transcript strand to get genomic orientation.
    """
    run_rsat("matrix-scan",
             ["-v", 1,
              "-i", fasta_path,
              "-m", matrix_path, "-matrix_format", "transfac",
              "-bgfile", bg_path,
              "-bg_pseudo", BG_PSEUDO,
              "-pseudo", MATRIX_PSEUDO,
              SCAN_STRANDS,
              "-origin", "-0",
              "-return", "sites,pval,limits",
              "-uth", f"pval {PVAL_THRESHOLD}",
              "-o", out_path])
    return out_path


SCAN_SPIC = RESULTS / "allpromotors_regulation_matrix_scan.ft.txt"
matrix_scan(FASTA_SPIC, MATRIX_FILE, BG_SPIC, SCAN_SPIC)
print("scan written ->", SCAN_SPIC)

In [1]:
#| label: rsat_scan_summary
#| code-fold: true
#| code-summary: "Hits per motif"
#| output: asis

hits = pd.read_csv(SCAN_SPIC, sep="\t", comment=";")
sites_all = hits[hits["ft_type"] == "site"]

n_seqs = sum(1 for line in Path(FASTA_SPIC).read_text().splitlines()
             if line.startswith(">"))
n_hits = len(sites_all)
n_prom = sites_all["#seq_id"].nunique()

summary = (sites_all.groupby("ft_name")
           .agg(sites=("ft_name", "size"),
                promoters=("#seq_id", "nunique"),
                best_pval=("Pval", "min"))
           .reset_index())

display(Markdown(
    f"**{n_hits} sites** at *P* < {PVAL_THRESHOLD:g} across **{n_prom}** of "
    f"**{n_seqs}** promoters scanned."
))
summary

NameError: name 'pd' is not defined

**A note on matrix versions.** The matrices used here are from **CIS-BP 2.00**, the release bundled with RSAT. The SOP cites **CIS-BP 3.00** accessions (`T176529_3.00`, matrix `M03213_3.00`), which are not present in the 2.00 collection. Scores and *P*-values are therefore not directly comparable with the earlier Web-form results, and the CIS-BP release should be stated explicitly in any methods section: the accession `M03213` alone is ambiguous across the two numbering schemes.

The two SpiC matrices in 2.00 are `M01480_2.00` (9 columns) and `M03009_2.00` (14 columns). The shorter one is expected to match too permissively for the same reason the shorter 3.00 motif was discarded, so `M03009_2.00` is used here — though this should be verified against the data rather than assumed.

## 4.2. Article ready table

For publication, the motif coordinates were matched to the chromosomal regions of the transcript promoters.

### 4.2.1 SpiC regulatory network

In [39]:
#| label: rsat_spic_load
#| code-fold: true
#| code-summary: "Load and Clean RSAT Motif Hits (SpiC Network)"
#| output: false
#open and clean data

Spic_genes = pd.read_csv("/srv/ext_home/ssalazar/projects/GIMM_projects/DNA_018_PromotorSeq_MotifFinder/Results/allpromotors_regulation_matrix_scan.ft.txt", sep="\t", comment=";")           # ';' lines are RSAT metadata
#fts = pd.read_csv("/srv/ext_home/ssalazar/projects/GIMM_projects/DNA_018_PromotorSeq_MotifFinder/Results/FT_cross_regulation_matrix_scan.ft.txt", sep="\t", comment=";")           # ';' lines are RSAT metadata

# prepare data from RSAT tools 
sites = Spic_genes[Spic_genes["ft_type"] == "site"].copy()                 # drop the 'limit' (START_END) rows, not significant

sites[["Gene_Name", "TranscriptID"]] = sites["#seq_id"].str.split("|", expand=True)  # seq_id equals the fasta header given: Gene_Name|TranscriptID
            # origin=end -> TSS sits at -201

sites_filtered =sites[['Gene_Name', 'TranscriptID', 'sequence', 'start', 'end', 'strand', 'Pval', 'sig']].copy()
sites_filtered = sites_filtered.rename(columns={'sequence': 'Motif_Sequence_Found',
                                                'start':   "start_motif_pos_promotor",   # 1-based TSS base
                                                'end': "end_motif_pos_promotor",
                                                'strand': "strand_motif"})

#get info from transcript
meta_transcript = (gtf_sub.df[["Chromosome", "Strand", "transcript_id"]]
                   .rename(columns={"Strand": "Chromosome_Strand",
                                    "transcript_id": "TranscriptID"}))

# one lookup from five_end(); gives the start of the transcript (TSS) in genomic coordinates, 1-based, strand dependent
meta = (gtf_sub.five_end().df
        [["transcript_id", "End"]]
        .rename(columns={"End":    "TSS_chromossomal_localization",   # 1-based TSS base
                         "transcript_id": "TranscriptID"}))

In [40]:
#| label: rsat_spic_coords
#| code-fold: true
#| code-summary: "Map SpiC Motif Hits to Genomic Coordinates"
#| output: false

meta = meta_transcript.merge(meta, on="TranscriptID", how="left")
sites_filtered = sites_filtered.merge(meta, on="TranscriptID", how="left")

# transcript-relative edges: 0 = TSS base, + = downstream (origin=end anchor at -201)

sites_filtered["start_motif_pos_promotor"] = sites_filtered["start_motif_pos_promotor"].astype(int) + 201
sites_filtered["end_motif_pos_promotor"]   = sites_filtered["end_motif_pos_promotor"].astype(int)   + 201

# nearest edge: end if upstream, start if downstream, 0 if it straddles the TSS
sites_filtered["dist_to_TSS"] = np.where(
    sites_filtered["end_motif_pos_promotor"] < 0, sites_filtered["end_motif_pos_promotor"],
    np.where(sites_filtered["start_motif_pos_promotor"] > 0, sites_filtered["start_motif_pos_promotor"], 0),
)

# to genome (1-based): + adds, - subtracts (edges swap so start <= end)
plus = sites_filtered["Chromosome_Strand"].eq("+")                    # was sites_filtered["Strand"]  -> KeyError
TSS  = sites_filtered["TSS_chromossomal_localization"]                # was sites_filtered["TSS"]     -> KeyError
sites_filtered["motif_genomic_start"] = np.where(plus, TSS + sites_filtered["start_motif_pos_promotor"],
                                                TSS - sites_filtered["end_motif_pos_promotor"]).astype(int)
sites_filtered["motif_genomic_end"]   = np.where(plus, TSS + sites_filtered["end_motif_pos_promotor"],
                                                TSS - sites_filtered["start_motif_pos_promotor"]).astype(int)
sites_filtered["motif_genomic_localization"] = (
    sites_filtered["motif_genomic_start"].astype(str)
    + "-"
    + sites_filtered["motif_genomic_end"].astype(str)
)
sense = sites_filtered["strand_motif"].eq("D")
sites_filtered["motif_genomic_strand"] = np.where(plus == sense, "+", "-")

sites_filtered.drop(columns=["strand_motif", "motif_genomic_end", "motif_genomic_start"], inplace=True)

sites_filtered.to_csv("/srv/ext_home/ssalazar/projects/GIMM_projects/DNA_018_PromotorSeq_MotifFinder/Results/SpiC_allpromoters_motifs_coordinates.csv", index=False)

In [41]:
#| label: spic_motifs_table
#| code-fold: true
#| code-summary: "SpiC Network Motif Coordinates"
#| output: asis
sites_filtered    # Inspect the SpiC-network motif hits mapped to genomic coordinates


,Gene_Name,TranscriptID,Motif_Sequence_Found,start_motif_pos_promotor,end_motif_pos_promotor,Pval,sig,Chromosome,Chromosome_Strand,TSS_chromossomal_localization,dist_to_TSS,motif_genomic_localization,motif_genomic_strand
0,Slc40a1,ENSMUST00000027137,AAAATGGGGTAGTA,-1110,-1097,0.000020,4.690,1,-,45964818,-1097,45965915-45965928,+
1,Slc40a1,ENSMUST00000027137,GTGACTGAGCTATG,-1789,-1776,0.000054,4.265,1,-,45964818,-1776,45966594-45966607,-
2,Blvra,ENSMUST00000002064,ATGAATAAGCCATA,-994,-981,0.000075,4.128,2,+,126912567,-981,126911573-126911586,-
3,Cat,ENSMUST00000028610,GAACGTGAGACAGCA,77,91,0.000007,5.156,2,-,103315505,77,103315414-103315428,+
4,Cat,ENSMUST00000028610,GTGACTGAGGACAT,-979,-966,0.000066,4.177,2,-,103315505,-966,103316471-103316484,+
...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,Acsl4,ENSMUST00000112904,GCGCGTCACTCAGCA,-295,-281,0.000048,4.323,X,-,141173789,-281,141174070-141174084,+
59,Acsl4,ENSMUST00000112904,AAAAAGAAGAAGAA,-962,-949,0.000085,4.073,X,-,141173789,-949,141174738-141174751,+
60,Nox1,ENSMUST00000033610,TAAACGAGGAAGTC,-900,-887,0.000026,4.581,X,-,133012603,-887,133013490-133013503,-
61,Nox1,ENSMUST00000033610,TTAAAGAGGAACTT,-942,-929,0.000048,4.315,X,-,133012603,-929,133013532-133013545,-


### 4.2.2 Transcription Factors Cross Regulation

In [42]:
#| label: rsat_ft_load
#| code-fold: true
#| code-summary: "Load and Clean RSAT Motif Hits (TF Cross-Regulation)"
#| output: false
#open and clean data

ft_genes = pd.read_csv("/srv/ext_home/ssalazar/projects/GIMM_projects/DNA_018_PromotorSeq_MotifFinder/Results/ft_cross_regulation_matrix_scan.ft.txt", sep="\t", comment=";")           # ';' lines are RSAT metadata
#fts = pd.read_csv("/srv/ext_home/ssalazar/projects/GIMM_projects/DNA_018_PromotorSeq_MotifFinder/Results/FT_cross_regulation_matrix_scan.ft.txt", sep="\t", comment=";")           # ';' lines are RSAT metadata

# prepare data from RSAT tools 
sites = ft_genes[ft_genes["ft_type"] == "site"].copy()                 # drop the 'limit' (START_END) rows, not significant

sites[["Gene_Name", "TranscriptID"]] = sites["#seq_id"].str.split("|", expand=True)  # seq_id equals the fasta header given: Gene_Name|TranscriptID
            # origin=end -> TSS sits at -201

sites_filtered =sites[['Gene_Name', 'TranscriptID', 'sequence', 'start', 'end', 'strand', 'Pval', 'sig']].copy()
sites_filtered = sites_filtered.rename(columns={'sequence': 'Motif_Sequence_Found',
                                                'start':   "start_motif_pos_promotor",   # 1-based TSS base
                                                'end': "end_motif_pos_promotor",
                                                'strand': "strand_motif"})

#get info from transcript
meta_transcript = (gtf_sub_ft.df[["Chromosome", "Strand", "transcript_id"]]
                   .rename(columns={"Strand": "Chromosome_Strand",
                                    "transcript_id": "TranscriptID"}))

# one lookup from five_end(); gives the start of the transcript (TSS) in genomic coordinates, 1-based, strand dependent
meta = (gtf_sub_ft.five_end().df
        [["transcript_id", "End"]]
        .rename(columns={"End":    "TSS_chromossomal_localization",   # 1-based TSS base
                         "transcript_id": "TranscriptID"}))
print(meta)

         TranscriptID  TSS_chromossomal_localization
0  ENSMUST00000102672                       75534985
1  ENSMUST00000004473                       88518885
2  ENSMUST00000026703                       87495782


In [43]:
#| label: rsat_ft_coords
#| code-fold: true
#| code-summary: "Map TF Motif Hits to Genomic Coordinates"
#| output: false

meta = meta_transcript.merge(meta, on="TranscriptID", how="left")
sites_filtered = sites_filtered.merge(meta, on="TranscriptID", how="left")

# transcript-relative edges: 0 = TSS base, + = downstream (origin=end anchor at -201)

sites_filtered["start_motif_pos_promotor"] = sites_filtered["start_motif_pos_promotor"].astype(int) + 201
sites_filtered["end_motif_pos_promotor"]   = sites_filtered["end_motif_pos_promotor"].astype(int)   + 201

# nearest edge: end if upstream, start if downstream, 0 if it straddles the TSS
sites_filtered["dist_to_TSS"] = np.where(
    sites_filtered["end_motif_pos_promotor"] < 0, sites_filtered["end_motif_pos_promotor"],
    np.where(sites_filtered["start_motif_pos_promotor"] > 0, sites_filtered["start_motif_pos_promotor"], 0),
)

# to genome (1-based): + adds, - subtracts (edges swap so start <= end)
plus = sites_filtered["Chromosome_Strand"].eq("+")                    # was sites_filtered["Strand"]  -> KeyError
TSS  = sites_filtered["TSS_chromossomal_localization"]                # was sites_filtered["TSS"]     -> KeyError
sites_filtered["motif_genomic_start"] = np.where(plus, TSS + sites_filtered["start_motif_pos_promotor"],
                                                TSS - sites_filtered["end_motif_pos_promotor"]).astype(int)
sites_filtered["motif_genomic_end"]   = np.where(plus, TSS + sites_filtered["end_motif_pos_promotor"],
                                                TSS - sites_filtered["start_motif_pos_promotor"]).astype(int)

sites_filtered["motif_genomic_localization"] = (
    sites_filtered["motif_genomic_start"].astype(str)
    
    + "-"
    + sites_filtered["motif_genomic_end"].astype(str)
)
sense = sites_filtered["strand_motif"].eq("D")
sites_filtered["motif_genomic_strand"] = np.where(plus == sense, "+", "-")

sites_filtered.drop(columns=["strand_motif", "motif_genomic_end", "motif_genomic_start"], inplace=True)


sites_filtered.to_csv("/srv/ext_home/ssalazar/projects/GIMM_projects/DNA_018_PromotorSeq_MotifFinder/Results/FT_cross_regulation_motifs_coordinates.csv", index=False)

In [44]:
#| label: ft_motifs_table
#| code-fold: true
#| code-summary: "TF Cross-Regulation Motif Coordinates"
#| output: asis
sites_filtered    # Inspect the TF cross-regulation motif hits mapped to genomic coordinates


,Gene_Name,TranscriptID,Motif_Sequence_Found,start_motif_pos_promotor,end_motif_pos_promotor,Pval,sig,Chromosome,Chromosome_Strand,TSS_chromossomal_localization,dist_to_TSS,motif_genomic_localization,motif_genomic_strand
0,Nfe2l2,ENSMUST00000102672,ACAAGGAGGAAGTA,-1944,-1931,0.000002,5.637,2,-,75534985,-1931,75536916-75536929,-
1,Nfe2l2,ENSMUST00000102672,AAATAGAGTAAGTA,-841,-828,0.000014,4.843,2,-,75534985,-828,75535813-75535826,-
2,Nfe2l2,ENSMUST00000102672,CAAATCAGGGAGTG,-475,-462,0.000088,4.056,2,-,75534985,-462,75535447-75535460,-
3,Spic,ENSMUST00000004473,ATGACTCAGAAGTT,-787,-774,0.000005,5.282,10,-,88518885,-774,88519659-88519672,+
4,Spic,ENSMUST00000004473,ATGATTCAGCTTTA,179,192,0.000047,4.326,10,-,88518885,179,88518693-88518706,+
5,Spic,ENSMUST00000004473,GGAGATGACTCAGAA,-784,-770,0.000100,3.992,10,-,88518885,-770,88519655-88519669,+


# 5. Conclusion

This report established a reproducible workflow for extracting promoter sequences and mapping transcription-factor binding motifs in support of the SpiC regulatory analysis, working from the *Mus musculus* GRCm39 reference genome (Ensembl release 116) via `pyranges`.

Promoter regions were defined as −2,000 bp to +200 bp relative to the TSS of each canonical transcript and extracted in a strand-aware manner for two sets of genes: (i) the panel of SpiC-regulated candidate genes and (ii) the *Spic* / *Nfe2l2* pair used to probe direct cross-regulation. The resulting sequences were written to FASTA files, scanned for SpiC and Nfe2l2 motifs with RSAT, and the significant hits were translated from promoter-relative coordinates back to genomic coordinates to build publication-ready tables.

Detection of SpiC and Nfe2l2 motifs within these promoters provides preliminary, sequence-based evidence for potential direct regulatory interactions. Functional validation would be required to distinguish direct regulation from indirect, Nfe2l2-mediated regulation.


# **Session Information**

In [45]:
#| label: session_information
#| echo: false

## Report package versions and dependencies for reproducibility
print(session_info.show(html=False, dependencies=True))


-----
numpy               2.1.3
pandas              2.2.3
pyfaidx             0.9.0.4
pyranges            0.1.4
session_info        v1.0.1
-----
anyio                       NA
arrow                       1.3.0
astroid                     3.3.9
asttokens                   NA
attr                        25.3.0
attrs                       25.3.0
babel                       2.17.0
brotli                      1.1.0
certifi                     2026.06.17
charset_normalizer          3.4.1
colorama                    0.4.6
comm                        0.2.2
cython_runtime              NA
dateutil                    2.9.0.post0
debugpy                     1.8.14
decorator                   5.2.1
executing                   2.1.0
fastjsonschema              NA
fqdn                        NA
idna                        3.10
ipykernel                   6.29.5
ipywidgets                  8.1.5
isoduration                 NA
jaraco                      NA
jedi                        0.19.2
jinja2    

/srv/ext_home/ssalazar/.local/lib/python3.12/site-packages/session_info/main.py:213: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  mod_version = _find_version(mod.__version__)


In [46]:
#| label: render_html
#| echo: false
#| output: false
#| message: false

##render html
os.system(f'quarto render "/srv/ext_home/ssalazar/projects/GIMM_projects/DNA_018_PromotorSeq_MotifFinder/Notebooks/PromoterExtraction_adapted.ipynb" --to html')


pandoc 
  to: html
  output-file: PromoterExtraction_adapted.html
  standalone: true
  embed-resources: true
  section-divs: true
  html-math-method: mathjax
  wrap: none
  default-image-extension: png
  toc: true
  toc-depth: 5
  number-sections: true
  variables: {}
  
metadata
  document-css: false
  link-citations: true
  date-format: long
  lang: en
  title: DNA008 Report
  author: Andre Barros (adapted)
  date: last-modified
  editor: visual
  jupyter: python
  page-layout: article
  
Output created: PromoterExtraction_adapted.html



0